In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load master
df = pd.read_csv("../data/raw/civilisations_master.csv")

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

print("\n=== NULL CHECK ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found")

print("\n=== SAMPLE ROW ===")
print(df[df['name'] == 'Roman Empire'].T)

=== SHAPE ===
(112, 21)

=== COLUMNS ===
  civilisation_id: int64
  name: str
  region: str
  sub_region: str
  founded_year: int64
  peak_year: int64
  collapse_start_year: int64
  collapse_end_year: int64
  peak_population_m: float64
  peak_territory_km2: int64
  latitude: float64
  longitude: float64
  primary_collapse_trigger: str
  secondary_collapse_trigger: str
  conquered_by: str
  pressured_by: str
  succeeded_by: str
  contemporary_rivals: str
  notes: str
  collapse_duration_years: float64
  lifespan_years: float64

=== NULL CHECK ===
secondary_collapse_trigger    29
conquered_by                  50
pressured_by                  21
succeeded_by                  21
collapse_duration_years       67
lifespan_years                67
dtype: int64

=== SAMPLE ROW ===
                                                                            8
civilisation_id                                                             9
name                                                         

In [2]:
# ── 1. Strip whitespace from all string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

# ── 2. Replace empty strings with NaN for optional fields
optional_fields = [
    'secondary_collapse_trigger', 'conquered_by',
    'pressured_by', 'succeeded_by', 'contemporary_rivals'
]
df[optional_fields] = df[optional_fields].replace('', np.nan)

# ── 3. Validate year logic — catch any data entry errors
year_errors = []

for _, row in df.iterrows():
    if row['founded_year'] >= row['collapse_end_year']:
        year_errors.append(f"{row['name']}: founded {row['founded_year']} >= collapse_end {row['collapse_end_year']}")
    if row['collapse_start_year'] > row['collapse_end_year']:
        year_errors.append(f"{row['name']}: collapse_start {row['collapse_start_year']} > collapse_end {row['collapse_end_year']}")
    if row['peak_year'] < row['founded_year']:
        year_errors.append(f"{row['name']}: peak {row['peak_year']} before founding {row['founded_year']}")
    if row['peak_year'] > row['collapse_end_year']:
        year_errors.append(f"{row['name']}: peak {row['peak_year']} after collapse_end {row['collapse_end_year']}")

if year_errors:
    print("⚠ Year logic errors found:")
    for e in year_errors:
        print(f"  {e}")
else:
    print("✓ All year logic checks passed")

# ── 4. Validate coordinates are plausible
coord_errors = df[
    (df['latitude'] < -90) | (df['latitude'] > 90) |
    (df['longitude'] < -180) | (df['longitude'] > 180)
]
if len(coord_errors) > 0:
    print(f"⚠ Coordinate errors: {coord_errors['name'].tolist()}")
else:
    print("✓ All coordinates valid")

print(f"\n✓ Cleaned dataframe: {df.shape[0]} rows, {df.shape[1]} columns")

✓ All year logic checks passed
✓ All coordinates valid

✓ Cleaned dataframe: 112 rows, 21 columns


In [3]:
# ── FEATURE GROUP 1: Time-based features

# Rise duration: founding to peak
df['rise_duration_years'] = df['peak_year'] - df['founded_year']

# Peak plateau: how long at peak before decline started
df['peak_plateau_years'] = df['collapse_start_year'] - df['peak_year']

# Collapse duration already exists, recalculate cleanly
df['collapse_duration_years'] = df['collapse_end_year'] - df['collapse_start_year']

# Total lifespan
df['lifespan_years'] = df['collapse_end_year'] - df['founded_year']

# Rise-to-fall ratio: >1 means rose faster than it fell, <1 means fell faster than it rose
df['rise_to_fall_ratio'] = df['rise_duration_years'] / df['collapse_duration_years'].replace(0, 1)

# Collapse speed: territory lost per year (proxy using peak territory)
df['collapse_speed'] = df['peak_territory_km2'] / df['collapse_duration_years'].replace(0, 1)


# ── FEATURE GROUP 2: Scale features

# Log-transform population and territory (right-skewed distributions)
df['log_peak_population'] = np.log1p(df['peak_population_m'])
df['log_peak_territory'] = np.log1p(df['peak_territory_km2'])

# Territory per capita at peak (proxy for resource intensity)
df['territory_per_capita'] = df['peak_territory_km2'] / (df['peak_population_m'] * 1e6)


# ── FEATURE GROUP 3: Era classification
def classify_era(year):
    if year < -1200:
        return 'Bronze Age'
    elif year < 500:
        return 'Classical'
    elif year < 1500:
        return 'Medieval'
    elif year < 1800:
        return 'Early Modern'
    elif year < 1914:
        return 'Industrial'
    else:
        return 'Modern'

df['founding_era'] = df['founded_year'].apply(classify_era)
df['collapse_era'] = df['collapse_end_year'].apply(classify_era)


# ── FEATURE GROUP 4: Interaction complexity
# Count number of rivals, pressuring empires, successors
df['n_contemporary_rivals'] = df['contemporary_rivals'].fillna('').apply(
    lambda x: len(x.split('|')) if x else 0
)
df['n_pressuring_empires'] = df['pressured_by'].fillna('').apply(
    lambda x: len(x.split('|')) if x else 0
)
df['n_successors'] = df['succeeded_by'].fillna('').apply(
    lambda x: len(x.split('|')) if x else 0
)
df['was_conquered'] = df['conquered_by'].notna().astype(int)


# ── FEATURE GROUP 5: Normalised trajectory features (for Pillar 5 ML)
# These express each feature as a percentile rank across all civilisations
# so the logistic regression isn't dominated by scale differences

from scipy.stats import rankdata

scale_features = [
    'lifespan_years', 'rise_duration_years', 'collapse_duration_years',
    'peak_population_m', 'peak_territory_km2', 'n_contemporary_rivals',
    'rise_to_fall_ratio'
]

for feat in scale_features:
    df[f'{feat}_pct'] = rankdata(df[feat]) / len(df)


print("=== ENGINEERED FEATURES SUMMARY ===")
print(f"\nTotal features: {df.shape[1]}")

print("\n── Time features ──")
time_cols = ['rise_duration_years', 'peak_plateau_years',
             'collapse_duration_years', 'lifespan_years', 'rise_to_fall_ratio']
print(df[time_cols].describe().round(1))

print("\n── Scale features ──")
scale_cols = ['peak_population_m', 'peak_territory_km2',
              'log_peak_population', 'log_peak_territory']
print(df[scale_cols].describe().round(2))

print("\n── Era breakdown ──")
print(df['founding_era'].value_counts())

print("\n── Interaction complexity ──")
print(df[['n_contemporary_rivals', 'n_pressuring_empires',
          'n_successors', 'was_conquered']].describe().round(2))

=== ENGINEERED FEATURES SUMMARY ===

Total features: 41

── Time features ──
       rise_duration_years  peak_plateau_years  collapse_duration_years  \
count                112.0               112.0                    112.0   
mean                 218.6               134.3                     61.3   
std                  193.0               146.9                     75.0   
min                    6.0                 0.0                      1.0   
25%                   70.8                40.0                     11.8   
50%                  155.5               100.0                     36.5   
75%                  328.8               166.2                     79.5   
max                  900.0              1000.0                    400.0   

       lifespan_years  rise_to_fall_ratio  
count           112.0               112.0  
mean            414.2                16.7  
std             319.0                37.3  
min              11.0                 0.3  
25%             172.5      

In [4]:
from sklearn.preprocessing import LabelEncoder

# ── Encode primary collapse trigger as both label and one-hot
trigger_dummies = pd.get_dummies(
    df['primary_collapse_trigger'],
    prefix='trigger'
)
df = pd.concat([df, trigger_dummies], axis=1)

# ── Encode region as numeric label (for spatial analysis)
le_region = LabelEncoder()
df['region_encoded'] = le_region.fit_transform(df['region'])

# ── Encode founding era as ordered numeric
era_order = {
    'Bronze Age': 0,
    'Classical': 1,
    'Medieval': 2,
    'Early Modern': 3,
    'Industrial': 4,
    'Modern': 5
}
df['founding_era_numeric'] = df['founding_era'].map(era_order)
df['collapse_era_numeric'] = df['collapse_era'].map(era_order)

print("=== CATEGORICAL ENCODING ===")
print(f"\nTrigger dummy columns: {[c for c in df.columns if c.startswith('trigger_')]}")
print(f"\nRegion encoding:")
for i, label in enumerate(le_region.classes_):
    print(f"  {i}: {label}")
print(f"\nEra encoding: {era_order}")

=== CATEGORICAL ENCODING ===

Trigger dummy columns: ['trigger_climate', 'trigger_conquest', 'trigger_economic', 'trigger_fragmentation', 'trigger_migration', 'trigger_overextension']

Region encoding:
  0: Africa
  1: Americas
  2: Central Asia
  3: East Asia
  4: Europe
  5: Middle East
  6: South Asia

Era encoding: {'Bronze Age': 0, 'Classical': 1, 'Medieval': 2, 'Early Modern': 3, 'Industrial': 4, 'Modern': 5}


In [5]:
# Load Maddison data
maddison = pd.read_excel(
    "../data/raw/maddison_gdp.xlsx",
    sheet_name="Full data",
    engine="openpyxl"
)

print("Maddison columns:", maddison.columns.tolist())
print("Shape:", maddison.shape)
print("\nSample:")
print(maddison.head(3))

Maddison columns: ['countrycode', 'country', 'region', 'year', 'gdppc', 'pop']
Shape: (131144, 6)

Sample:
  countrycode      country                     region  year  gdppc  pop
0         AFG  Afghanistan  South and South East Asia     1    NaN  NaN
1         AFG  Afghanistan  South and South East Asia   730    NaN  NaN
2         AFG  Afghanistan  South and South East Asia  1000    NaN  NaN


In [6]:
# Map each civilisation to its closest Maddison country code
# This is a deliberate analytical choice — documented for the README
civilisation_maddison_map = {
    "Sumerian City-States":           "IRQ",
    "Akkadian Empire":                "IRQ",
    "Neo-Babylonian Empire":          "IRQ",
    "Achaemenid Persian Empire":      "IRN",
    "Sassanid Empire":                "IRN",
    "Ancient Egypt - New Kingdom":    "EGY",
    "Classical Greece":               "GRC",
    "Macedonian Empire":              "GRC",
    "Roman Empire":                   "ITA",
    "Byzantine Empire":               "TUR",
    "Parthian Empire":                "IRN",
    "Umayyad Caliphate":              "SAU",
    "Abbasid Caliphate":              "IRQ",
    "Ottoman Empire":                 "TUR",
    "Maurya Empire":                  "IND",
    "Gupta Empire":                   "IND",
    "Mughal Empire":                  "IND",
    "Qin Dynasty":                    "CHN",
    "Han Dynasty":                    "CHN",
    "Tang Dynasty":                   "CHN",
    "Song Dynasty":                   "CHN",
    "Yuan Dynasty":                   "CHN",
    "Ming Dynasty":                   "CHN",
    "Qing Dynasty":                   "CHN",
    "Mongol Empire":                  "MNG",
    "Timurid Empire":                 "UZB",
    "Kingdom of Kush":                "SDN",
    "Kingdom of Aksum":               "ETH",
    "Mali Empire":                    "MLI",
    "Classic Maya":                   "MEX",
    "Teotihuacan":                    "MEX",
    "Aztec Empire":                   "MEX",
    "Inca Empire":                    "PER",
    "Carolingian Empire":             "DEU",
    "Holy Roman Empire":              "DEU",
    "Spanish Empire":                 "ESP",
    "Napoleonic Empire":              "FRA",
    "Austro-Hungarian Empire":        "AUT",
    "German Empire and Third Reich":  "DEU",
    "Italian Colonial Empire":        "ITA",
    "Empire of Japan":                "JPN",
    "French Colonial Empire":         "FRA",
    "British Empire":                 "GBR",
    "Russian Empire and USSR":        "RUS",
    "Portuguese Empire":              "PRT",
}

df['maddison_country_code'] = df['name'].map(civilisation_maddison_map)

# Verify all civilisations got mapped
unmapped = df[df['maddison_country_code'].isna()]['name'].tolist()
if unmapped:
    print(f"⚠ Unmapped civilisations: {unmapped}")
else:
    print(f"✓ All 45 civilisations mapped to Maddison country codes")

# Extract peak-year GDP and population for each civilisation
# Maddison only goes back to 1 CE reliably, so BCE civilisations get NaN — expected
records = []

for _, row in df.iterrows():
    code = row['maddison_country_code']
    peak = row['peak_year']

    # Filter Maddison for this country
    country_data = maddison[maddison['countrycode'] == code].copy()

    if len(country_data) == 0:
        records.append({'civilisation_id': row['civilisation_id'],
                        'maddison_gdppc_at_peak': np.nan,
                        'maddison_pop_at_peak': np.nan})
        continue

    # Find closest available year to peak year
    # Maddison doesn't have every year — find nearest
    available_years = country_data['year'].values
    if len(available_years) == 0:
        records.append({'civilisation_id': row['civilisation_id'],
                        'maddison_gdppc_at_peak': np.nan,
                        'maddison_pop_at_peak': np.nan})
        continue

    closest_year = available_years[np.argmin(np.abs(available_years - peak))]
    year_gap = abs(closest_year - peak)

    row_data = country_data[country_data['year'] == closest_year].iloc[0]

    records.append({
        'civilisation_id': row['civilisation_id'],
        'maddison_gdppc_at_peak': row_data['gdppc'] if year_gap <= 200 else np.nan,
        'maddison_pop_at_peak': row_data['pop'] if year_gap <= 200 else np.nan,
        'maddison_year_used': closest_year,
        'maddison_year_gap': year_gap
    })

maddison_features = pd.DataFrame(records)
df = df.merge(maddison_features, on='civilisation_id', how='left')

# Summary
matched = df['maddison_gdppc_at_peak'].notna().sum()
print(f"\nMaddison GDP matched: {matched}/45 civilisations")
print(f"Unmatched (BCE or sparse data — expected): {45 - matched}")
print(f"\nGDP per capita range where available:")
print(df['maddison_gdppc_at_peak'].describe().round(1))

⚠ Unmapped civilisations: ['Indus Valley Civilisation', 'Kushan Empire', 'Chola Dynasty', 'Delhi Sultanate', 'Vijayanagara Empire', 'Maratha Empire', 'Ghana Empire', 'Kanem-Bornu Empire', 'Kingdom of Kongo', 'Mutapa Empire', 'Ethiopian Empire', 'Sokoto Caliphate', 'Kingdom of Benin', 'Kilwa Sultanate', 'Asante Empire', 'Hittite Empire', 'Mycenaean Greece', 'Carthage', 'Seleucid Empire', 'Shang Dynasty', 'Zhou Dynasty', 'Sui Dynasty', 'Khmer Empire', 'Srivijaya Empire', 'Kievan Rus', 'Kingdom of France', 'Republic of Venice', 'Xiongnu Confederation', 'Golden Horde', 'Goryeo', 'Silla', 'Koguryo', 'Balhae', 'Yamato', 'Tokugawa Shogunate', 'Liao Dynasty', 'Jin Empire', 'Xixia', 'Pagan Empire', 'Ayutthaya', 'Champa', 'Dai Viet', 'Majapahit', 'Ilkhanate', 'Safavid Empire', 'Fatimid Caliphate', 'Seljuk Empire', 'Chagatai Khanate', 'Ghaznavid Emirate', 'Almohad Caliphate', 'Almoravid Dynasty', 'Ptolemaic Kingdom', 'Visigothic Kingdom', 'Angevin Empire', 'Great Moravia', 'Polish-Lithuanian Comm

In [7]:
# ── Final column audit before saving
analysis_cols = [
    # Identity
    'civilisation_id', 'name', 'region', 'sub_region',

    # Core years
    'founded_year', 'peak_year', 'collapse_start_year', 'collapse_end_year',

    # Geography
    'latitude', 'longitude',

    # Raw scale
    'peak_population_m', 'peak_territory_km2',

    # Collapse classification
    'primary_collapse_trigger', 'secondary_collapse_trigger',
    'conquered_by', 'pressured_by', 'succeeded_by', 'contemporary_rivals',

    # Engineered time features
    'rise_duration_years', 'peak_plateau_years',
    'collapse_duration_years', 'lifespan_years', 'rise_to_fall_ratio',
    'collapse_speed',

    # Engineered scale features
    'log_peak_population', 'log_peak_territory', 'territory_per_capita',

    # Era
    'founding_era', 'collapse_era',
    'founding_era_numeric', 'collapse_era_numeric',

    # Interaction complexity
    'n_contemporary_rivals', 'n_pressuring_empires',
    'n_successors', 'was_conquered',

    # Percentile ranks
    'lifespan_years_pct', 'rise_duration_years_pct',
    'collapse_duration_years_pct', 'peak_population_m_pct',
    'peak_territory_km2_pct', 'n_contemporary_rivals_pct',
    'rise_to_fall_ratio_pct',

    # Trigger dummies
    'trigger_climate', 'trigger_conquest', 'trigger_economic',
    'trigger_fragmentation', 'trigger_migration', 'trigger_overextension',

    # Encodings
    'region_encoded',

    # Maddison
    'maddison_country_code', 'maddison_gdppc_at_peak',
    'maddison_pop_at_peak', 'maddison_year_gap',

    # Notes
    'notes'
]

# Keep only analysis columns
df_clean = df[analysis_cols].copy()

# Save
import os
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv("../data/processed/civilisations_clean.csv", index=False)

print("=== FINAL CLEAN DATASET ===")
print(f"Shape: {df_clean.shape}")
print(f"File: data/processed/civilisations_clean.csv")
print(f"Size: {os.path.getsize('../data/processed/civilisations_clean.csv') / 1024:.1f} KB")

print("\n=== COLUMN GROUPS ===")
print(f"Identity columns:        4")
print(f"Core year columns:       4")
print(f"Geography columns:       2")
print(f"Raw scale columns:       2")
print(f"Collapse classification: 6")
print(f"Engineered time:         5")
print(f"Engineered scale:        3")
print(f"Era columns:             4")
print(f"Interaction complexity:  4")
print(f"Percentile ranks:        7")
print(f"Trigger dummies:         6")
print(f"Encodings:               1")
print(f"Maddison:                4")
print(f"Notes:                   1")
print(f"─────────────────────────")
print(f"Total:                  {df_clean.shape[1]}")

print("\n=== NULLS IN CLEAN FILE ===")
nulls = df_clean.isnull().sum()
nulls_present = nulls[nulls > 0]
if len(nulls_present) > 0:
    print(nulls_present)
    print("\n(NaNs in optional fields and Maddison columns are expected)")
else:
    print("No nulls")

print("\n=== READY FOR ANALYSIS ===")
print(df_clean[['name', 'lifespan_years', 'collapse_duration_years',
                'primary_collapse_trigger', 'founding_era']].to_string())

=== FINAL CLEAN DATASET ===
Shape: (112, 54)
File: data/processed/civilisations_clean.csv
Size: 59.5 KB

=== COLUMN GROUPS ===
Identity columns:        4
Core year columns:       4
Geography columns:       2
Raw scale columns:       2
Collapse classification: 6
Engineered time:         5
Engineered scale:        3
Era columns:             4
Interaction complexity:  4
Percentile ranks:        7
Trigger dummies:         6
Encodings:               1
Maddison:                4
Notes:                   1
─────────────────────────
Total:                  54

=== NULLS IN CLEAN FILE ===
secondary_collapse_trigger     29
conquered_by                   50
pressured_by                   21
succeeded_by                   21
maddison_country_code          67
maddison_gdppc_at_peak         95
maddison_pop_at_peak          101
maddison_year_gap              67
dtype: int64

(NaNs in optional fields and Maddison columns are expected)

=== READY FOR ANALYSIS ===
                               name  li